In [13]:
import kagglehub
import os

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.ensemble import VotingClassifier

from sklearn.model_selection import GridSearchCV

In [14]:
# 3.1.1 Data Source
path = kagglehub.dataset_download("abhi8923shriv/liver-disease-patient-dataset")
print("Path to dataset files:", path)

# Data Source - Using the path from kagglehub
csv_path = os.path.join(path, "Liver Patient Dataset (LPD)_train.csv")
df = pd.read_csv(csv_path, encoding='latin1')
display(df.head())

Using Colab cache for faster access to the 'liver-disease-patient-dataset' dataset.
Path to dataset files: /kaggle/input/liver-disease-patient-dataset


,Age of the patient,Gender of the patient,Total Bilirubin,Direct Bilirubin,Alkphos Alkaline Phosphotase,Sgpt Alamine Aminotransferase,Sgot Aspartate Aminotransferase,Total Protiens,ALB Albumin,A/G Ratio Albumin and Globulin Ratio,Result
0,65.0,Female,0.7,0.1,187.0,16.0,18.0,6.8,3.3,0.90,1
1,62.0,Male,10.9,5.5,699.0,64.0,100.0,7.5,3.2,0.74,1
2,62.0,Male,7.3,4.1,490.0,60.0,68.0,7.0,3.3,0.89,1
3,58.0,Male,1.0,0.4,182.0,14.0,20.0,6.8,3.4,1.00,1
4,72.0,Male,3.9,2.0,195.0,27.0,59.0,7.3,2.4,0.40,1


In [15]:
# 3.1.2 Data Types
print("Dataset Features Type : \n", df.dtypes)

Dataset Features Type : 
 Age of the patient                      float64
Gender of the patient                    object
Total Bilirubin                         float64
Direct Bilirubin                        float64
 Alkphos Alkaline Phosphotase           float64
 Sgpt Alamine Aminotransferase          float64
Sgot Aspartate Aminotransferase         float64
Total Protiens                          float64
 ALB Albumin                            float64
A/G Ratio Albumin and Globulin Ratio    float64
Result                                    int64
dtype: object


In [16]:
# 3.1.3 Manual Feature Selection
# Gender
if 'Gender of the patient' in df.columns:
    df = df.drop(columns=['Gender of the patient'], axis=1)
    print("Column 'Gender of the patient' dropped.")
else:
    print("Column 'Gender of the patient' already dropped or not found.")

Column 'Gender of the patient' dropped.


In [17]:
# Feature Description
print("Features Description:")
print(df.columns[0], "= Age of the patient (years)")
print(df.columns[1], "= Total bilirubin level in blood (mg/dL)")
print(df.columns[2], "= Conjugated bilirubin level (mg/dL)")
print(df.columns[3], "= Alkaline phosphatase enzyme level (IU/L)")
print(df.columns[4], "= ALT enzyme level (IU/L)")
print(df.columns[5], "= AST enzyme level (IU/L)")
print(df.columns[6], "= Total protein level in blood (g/dL)")
print(df.columns[7], "= Albumin level in blood (g/dL)")
print(df.columns[8], "= Albumin to globulin ratio")
print(df.columns[9], "= Target label (class)")
print("From (1 -> Liver Patient), (2 -> Non-Liver Patient)")
print("Change (1 -> Liver Patient = 1), (0 -> Non-Liver Patient)")

# No liver disease then:=0 for having liver disease then:=1
df['Result'] = df['Result'].map({1: 1, 2: 0})

Features Description:
Age of the patient = Age of the patient (years)
Total Bilirubin = Total bilirubin level in blood (mg/dL)
Direct Bilirubin = Conjugated bilirubin level (mg/dL)
 Alkphos Alkaline Phosphotase = Alkaline phosphatase enzyme level (IU/L)
 Sgpt Alamine Aminotransferase = ALT enzyme level (IU/L)
Sgot Aspartate Aminotransferase = AST enzyme level (IU/L)
Total Protiens = Total protein level in blood (g/dL)
 ALB Albumin = Albumin level in blood (g/dL)
A/G Ratio Albumin and Globulin Ratio = Albumin to globulin ratio
Result = Target label (class)
From (1 -> Liver Patient), (2 -> Non-Liver Patient)
Change (1 -> Liver Patient = 1), (0 -> Non-Liver Patient)


In [18]:
# 3.1.4 Renaming The Columns
print("Renaming The Columns :\n")
df = df.rename(columns={
  df.columns[0]: 'Age',
  df.columns[1]: 'TB',
  df.columns[2]: 'DB',
  df.columns[3]: 'Alkphos',
  df.columns[4]: 'Sgpt',
  df.columns[5]: 'Sgot',
  df.columns[6]: 'TP',
  df.columns[7]: 'ALB',
  df.columns[8]: 'A/G',
  })
df.head()

Renaming The Columns :



,Age,TB,DB,Alkphos,Sgpt,Sgot,TP,ALB,A/G,Result
0,65.0,0.7,0.1,187.0,16.0,18.0,6.8,3.3,0.90,1
1,62.0,10.9,5.5,699.0,64.0,100.0,7.5,3.2,0.74,1
2,62.0,7.3,4.1,490.0,60.0,68.0,7.0,3.3,0.89,1
3,58.0,1.0,0.4,182.0,14.0,20.0,6.8,3.4,1.00,1
4,72.0,3.9,2.0,195.0,27.0,59.0,7.3,2.4,0.40,1


In [9]:
# Function to drop duplicate rows
def drop_duplicate_rows(df):
  print("Original DataFrame shape:", df.shape)
  duplicate_rows_df = df[df.duplicated()]
  print("Number of duplicate rows found: ", duplicate_rows_df.shape[0])

  # Dropping The Duplicate Rows
  df = df.drop_duplicates()
  print("DataFrame shape after dropping duplicates:", df.shape)
  return df

# Applying the function to drop duplicate rows
df = drop_duplicate_rows(df)

Original DataFrame shape: (30691, 10)
Number of duplicate rows found:  14127
DataFrame shape after dropping duplicates: (16564, 10)


In [ ]:
# Combined function to drop null values and reset index
def clean_and_reset_data(df):
  print("Starting null value handling and index reset:")

  # Drop null values logic
  initial_rows_before_drop = df.shape[0]
  df = df.dropna()
  final_rows_after_drop = df.shape[0]
  print(f"Number of rows before dropping nulls: {initial_rows_before_drop}")
  print(f"Number of rows after dropping nulls: {final_rows_after_drop}")
  print(f"Number of null rows dropped: {initial_rows_before_drop - final_rows_after_drop}")
  return df

# Applying the combined function
df = clean_and_reset_data(df)
display(df.describe())

Starting null value handling and index reset:
Number of rows before dropping nulls: 19368
Number of rows after dropping nulls: 16389
Number of null rows dropped: 2979


,Age of the patient,Total Bilirubin,Direct Bilirubin,Alkphos Alkaline Phosphotase,Sgpt Alamine Aminotransferase,Sgot Aspartate Aminotransferase,Total Protiens,ALB Albumin,A/G Ratio Albumin and Globulin Ratio,Result
count,16389.000000,16389.000000,16389.000000,16389.000000,16389.000000,16389.000000,16389.000000,16389.000000,16389.000000,16389.000000
mean,43.770517,3.360431,1.530429,290.826835,80.147294,111.367564,6.487705,3.136573,0.946612,1.283056
std,16.529487,6.208708,2.894558,240.945972,180.010180,280.665994,1.090549,0.794006,0.323337,0.450497
min,4.000000,0.400000,0.100000,63.000000,10.000000,10.000000,2.700000,0.900000,0.300000,1.000000
25%,32.000000,0.800000,0.200000,175.000000,23.000000,25.000000,5.800000,2.600000,0.700000,1.000000
50%,45.000000,1.000000,0.300000,209.000000,35.000000,42.000000,6.600000,3.100000,0.930000,1.000000
75%,55.000000,2.700000,1.300000,298.000000,62.000000,88.000000,7.200000,3.800000,1.100000,2.000000
max,90.000000,75.000000,19.700000,2110.000000,2000.000000,4929.000000,9.600000,5.500000,2.800000,2.000000


In [ ]:
# Show Distribution of Gender
plt.figure(figsize=(8, 4))
ax = sns.countplot(data=df, x='Gender', hue='Gender', palette='viridis', legend=False)
plt.title('Distribution of Gender')
plt.xlabel('Gender')
plt.ylabel('Count')

# Add count values on top of the bars
for p in ax.patches:
  ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='baseline', fontsize=8, color='black', xytext=(0, 5), textcoords='offset points')

plt.show()

In [ ]:
# Outliers Detection (Box Plot)
# Numerical columns for box plots
numerical_cols = ['Age', 'TB', 'DB', 'Alkphos', 'Sgpt', 'Sgot', 'TP', 'ALB', 'A/G']

# Create box plots for numerical features
fig, axes = plt.subplots(3, 3, figsize=(30, 20))
axes = axes.flatten() # Flatten the 2x5 array of axes for easy iteration

for i, col in enumerate(numerical_cols):
  sns.boxplot(y=df[col], ax=axes[i])
  axes[i].set_title(col)
  axes[i].set_ylabel('') # Remove y-label as it's redundant with title

plt.tight_layout()
plt.suptitle('Box Plots of Numerical Features', y=1.02, fontsize=16) # Add a super title
plt.show()

In [ ]:
# Plot Distribusi
# Numerical columns for distribution plots
numerical_cols = ['Age', 'TB', 'DB', 'Alkphos', 'Sgpt', 'Sgot', 'TP', 'ALB', 'A/G']

# Create distribution plots (histograms) for numerical features
fig, axes = plt.subplots(3, 3, figsize=(30, 20))
axes = axes.flatten() # Flatten the 3x3 array of axes for easy iteration

for i, col in enumerate(numerical_cols):
  sns.histplot(df[col], kde=True, ax=axes[i])
  axes[i].set_title(f'Distribution of {col}')
  axes[i].set_xlabel(col)
  axes[i].set_ylabel('Frequency')

plt.tight_layout()
plt.suptitle('Distribution Plots of Numerical Features', y=1.02, fontsize=16) # Add a super title
plt.show()

In [ ]:
# Univariate Analysis
df.describe()

In [ ]:
# Distribution plots for numerical features
fig, axes = plt.subplots(3, 3, figsize=(30, 20))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
  sns.histplot(df[col], kde=True, ax=axes[i])
  axes[i].set_title(f'Distribution of {col}')
  axes[i].set_xlabel(col)
  axes[i].set_ylabel('Frequency')

plt.tight_layout()
plt.suptitle('Distribution Plots of Numerical Features', y=1.02, fontsize=16)
plt.show()

In [ ]:
# Bar plot for 'Gender'
plt.figure(figsize=(8, 4))
ax_gender = sns.countplot(data=df, x='Gender', hue='Gender', palette='viridis', legend=False)
plt.title('Distribution of Gender')
plt.xlabel('Gender')
plt.ylabel('Count')
for p in ax_gender.patches:
  ax_gender.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='baseline', fontsize=8, color='black', xytext=(0, 5), textcoords='offset points')
plt.show()

In [ ]:
# Bar plot for 'Result'
plt.figure(figsize=(8, 4))
ax_result = sns.countplot(data=df, x='Result', hue='Result', palette='magma', legend=False)
plt.title('Distribution of Result (Target Variable)')
plt.xlabel('Result (0: Non Liver Patient, 1: Liver Patient)')
plt.ylabel('Count')
for p in ax_result.patches:
  ax_result.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='baseline', fontsize=8, color='black', xytext=(0, 5), textcoords='offset points')
plt.show()

In [ ]:
# Analysis Bivariate Heatmap
plt.figure(figsize=(12, 8))
# Gunakan numeric_only=True untuk mengabaikan kolom non-numerik seperti 'Gender'
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')

In [ ]:
# Analisis Univariat
def plot_histograms(df, columns_sets):
  for columns in columns_sets:
    fig, axs = plt.subplots(1, len(columns), figsize=(15,3))
    for col, ax in zip(columns, axs):
      df[col].plot(kind='hist', bins=20, ax=ax, title=col)
      ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.show()

# Bersihkan nama kolom di liver_df untuk menghapus spasi di awal/akhir dan spasi yang tidak terputus.
df.columns = df.columns.str.strip().str.replace('\xa0', '')

columns_sets = [
  # Set 1: Informasi dasar pasien dan kadar bilirubin
  ["Age", "TB", "DB"],
  # Set 2: Kadar enzim hati (penanda kerusakan hati)
  ["Alkphos", "Sgpt", "Sgot"],
  # Set 3: Protein darah dan rasio albumin-globulin
  ["TP", "ALB", "A/G"],
]
plot_histograms(df, columns_sets)

In [ ]:
# Analisis Bivariate
def plot_bivariate_analysis(df, column_pairs):
  for i in range(0, len(column_pairs), 3):
    fig, axs = plt.subplots(1, min(3, len(column_pairs) - i), figsize=(20, 4))
    # Handle cases where subplots returns a single axis object instead of an array
    if not isinstance(axs, np.ndarray):
      axs = [axs]
    for (x, y), ax in zip(column_pairs[i:i + 3], axs):
      sns.regplot(data=df, x=x, y=y, scatter_kws={'s': 32, 'alpha': 0.8}, ax=ax)
      ax.set_title(f'{x} vs {y}')
    plt.tight_layout()
    plt.show()

column_pairs = [
  ('Age', 'Result'),
  ('TB', 'Result'),
  ('DB', 'Result'),
  ('Alkphos', 'Result'),
  ('Sgpt', 'Result'),
  ('Sgot', 'Result'),
  ('TP', 'Result'),
  ('ALB', 'Result'),
  ('A/G', 'Result')
]

plot_bivariate_analysis(df, column_pairs)

In [ ]:
# Function for Data Normalization
def normalize_data(df, numerical_cols):
  print("Starting data normalization:")
  # Create a MinMaxScaler instance
  scaler = MinMaxScaler()

  # Apply Min-Max Scaling to the numerical columns
  df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
  print("Data normalization complete.")
  return df

# Define the numerical columns to normalize (excluding 'Result' as it's the target variable)
numerical_cols = ['Age', 'TB', 'DB', 'Alkphos', 'Sgpt', 'Sgot', 'TP', 'ALB', 'A/G']

# Applying the function
df = normalize_data(df, numerical_cols)

# Display the first few rows of the DataFrame with normalized data
display(df.head())

In [ ]:
# Feature Engineering
# One-hot encoding
pd.get_dummies(df['Gender'], prefix = 'Gender').head()

In [ ]:
# Function for one-hot encoding the 'Gender' column
def encode_gender_feature(df):
  if 'Gender_Female' not in df.columns:
    print("Performing one-hot encoding for 'Gender' column...")
    dummies = pd.get_dummies(df['Gender'], prefix='Gender')
    df = pd.concat([df, dummies.astype(int)], axis=1)
    print("Gender columns added and converted to integer.")
  else:
    print("Gender columns already exist, skipping one-hot encoding.")
  return df

# Applying the function
df = encode_gender_feature(df)
df.head()

In [ ]:
# Function to reorder gender-related columns
def reorder_gender_columns(df):
  cols = df.columns.tolist()
  if 'Gender' in cols:
    gender_idx = cols.index('Gender')
    move_cols = ['Gender_Female', 'Gender_Male']

    # Remove move_cols from current position if they exist
    for c in move_cols:
      if c in cols:
        cols.remove(c)

    # Insert move_cols next to 'Gender'
    new_cols = cols[:gender_idx + 1] + move_cols + cols[gender_idx + 1:]
    df = df[new_cols]
  else:
    print("'Gender' column not found, skipping specific reordering for gender-related columns.")
  return df

# Applying the function
df = reorder_gender_columns(df)
display(df.head())

In [ ]:
# Drop_columns
def drop_columns(df, columns_to_drop):
  for col in columns_to_drop:
    if col in df.columns:
      df = df.drop(columns=[col])
      print(f"Column '{col}' has been dropped.")
    else:
      print(f"Column '{col}' not found; it may have already been dropped.")
  return df

# Applying the function to drop the 'Gender' column
df = drop_columns(df, ['Gender'])
display(df.head())

In [ ]:
# Function for Univariate Chi-Square Feature Selection
def perform_chi_square_selection(df):
  X = df.drop(columns=['Result'])
  y = df['Result']

  bestfeatures = SelectKBest(score_func=chi2, k='all')
  fit = bestfeatures.fit(X, y)

  dfscores = pd.DataFrame(fit.scores_)
  dfcolumns = pd.DataFrame(X.columns)

  featureScores = pd.concat([dfcolumns, dfscores], axis=1)
  featureScores.columns = ['Features', 'Score']

  # Display scores in descending order
  print("Feature Scores (Chi-square):")
  return featureScores.sort_values(by='Score', ascending=False)

# Applying the function
chi_square_scores = perform_chi_square_selection(df)
display(chi_square_scores)

In [ ]:
# Function to display feature importance using GradientBoostingClassifier
def display_gb_feature_importance(X, y):
  model = GradientBoostingClassifier(random_state=42) # Added random_state for reproducibility
  model.fit(X, y)
  feat_importances = pd.Series(model.feature_importances_, index=X.columns)

  plt.figure(figsize=(10, 6))
  feat_importances.nlargest(len(X.columns)).plot(kind='barh') # Plot all features
  plt.title('Gradient Boosting Feature Importance')
  plt.xlabel('Feature Importance Score')
  plt.ylabel('Features')
  plt.tight_layout()
  plt.show()

# Prepare X and y for the function call
X = df.drop(columns=['Result'])
y = df['Result']

# Apply the function
display_gb_feature_importance(X, y)

In [ ]:
# Drop columns with low feature importance score
drop_columns(df.head(), ['Age', 'Gender_Female', 'Gender_Male'])

In [ ]:
# Split data into training and testing sets
X = df.drop('Result', axis=1)
y = df['Result']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Data split into training and testing sets:")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# Buatkan agar kodenya
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score

# Inisialisasi model
gb_model = GradientBoostingClassifier(random_state=42)

# Parameter grid yang akan diuji
param_grid = {
  'n_estimators': [100, 200],
  'learning_rate': [0.1, 0.05],
  'max_depth': [3, 4],
}

# Inisialisasi GridSearch
grid_search = GridSearchCV(
  estimator=gb_model,
  param_grid=param_grid,
  cv=5,
  scoring='accuracy',
  n_jobs=-1,
  verbose=1
)

# Training
grid_search.fit(X_train, y_train)

# Best parameter
print("Best Parameters:", grid_search.best_params_)
print("Best Cross Validation Score:", grid_search.best_score_)

# Display all grid search combinations and their scores
print("\nAll Grid Search Combinations and Scores:")
results = grid_search.cv_results_
for i in range(len(results['params'])):
    print(f"Combination {i+1}: {results['params'][i]}, Mean Test Score: {results['mean_test_score'][i]:.4f}")

# Evaluasi pada data test
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("\nTest Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
import time

def train_and_evaluate_gradient_boosting(X_train, y_train, X_test, y_test):
  # Gradient Boosting (GB)
  gb_model = GradientBoostingClassifier(
    loss = "log_loss",
    learning_rate = 0.1,
    n_estimators = 300,
    max_depth = 5,
  )

  # Training Time
  start_train_time = time.time()
  gb_model.fit(X_train, y_train)
  training_time = time.time() - start_train_time

  # Testing Time
  start_test_time = time.time()
  gb_predictions = gb_model.predict(X_test)
  testing_time = time.time() - start_test_time

  gb_accuracy = accuracy_score(y_test, gb_predictions)
  print("Training Time (Seconds) :", round(training_time, 4), "seconds")
  print("Testing Time (Seconds) :", round(testing_time, 4), "seconds")

  gb_classification_report = classification_report(y_test, gb_predictions)
  print("Gradient Boosting Classification Report :\n", gb_classification_report)
  print("Gradient Boosting Accuracy :", gb_accuracy)
  return gb_model, gb_predictions

# Applying the function
gb_model, gb_predictions = train_and_evaluate_gradient_boosting(X_train, y_train, X_test, y_test)

In [ ]:
gb_confusion_matrix = confusion_matrix(y_test, gb_predictions)
plt.figure(figsize=(8, 6))
sns.heatmap(gb_confusion_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=['Non-Liver', 'Liver'], yticklabels=['Non-Liver', 'Liver'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Gradient Boosting Confusion Matrix Heatmap')
plt.show()